In [2]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


# Patients With a Condition

## Problem Description

You are given a PySpark DataFrame named **`patients`** containing information about patients and their medical condition codes.

The `conditions` column contains multiple condition codes separated by spaces.

Your task is to find all patients who have **Type I Diabetes**.

A Type I Diabetes condition code always starts with the prefix **`DIAB1`**.

A patient should be included if any condition code starts with `DIAB1`.

### Input DataFrame

The `patients` DataFrame contains:

| Column | Data Type | Description |
|---|---|---|
| `patient_id` | Integer | Unique identifier for the patient |
| `patient_name` | String | Name of the patient |
| `conditions` | String | Space-separated list of condition codes |

### Expected Output

Return the following columns:

- `patient_id`
- `patient_name`
- `conditions`

### Example

| patient_id | patient_name | conditions |
|---:|---|---|
| 1 | Daniel | YFEV COUGH |
| 2 | Alice | DIAB1 MYOP |
| 3 | Bob | ACNE DIAB100 |
| 4 | George | FLU |

Expected result:

| patient_id | patient_name | conditions |
|---:|---|---|
| 2 | Alice | DIAB1 MYOP |
| 3 | Bob | ACNE DIAB100 |

### Problem Pattern

**Filtering → String Matching → Regular Expression**

A useful regex pattern is:

`(^| )DIAB1`

This matches `DIAB1` when it appears at the beginning of the string or immediately after a space.

In [3]:
data = [
    (1, "Daniel", "YFEV COUGH"),
    (2, "Alice", "DIAB1 MYOP"),
    (3, "Bob", "ACNE DIAB100"),
    (4, "George", "FLU"),
    (5, "Marta", "DIAB150 ACNE FLU"),
    (6, "John", "HIGH_BP ASTHMA"),
    (7, "Sarah", "DIAB2 FEVER"),
    (8, "Robert", "COLD DIAB101"),
    (9, "Emma", "HEART ACNE"),
    (10, "David", "DIAB10 COUGH")
]

columns = [
    "patient_id",
    "patient_name",
    "conditions"
]

patients = spark.createDataFrame(data, columns)

patients.show(truncate=False)

+----------+------------+----------------+
|patient_id|patient_name|conditions      |
+----------+------------+----------------+
|1         |Daniel      |YFEV COUGH      |
|2         |Alice       |DIAB1 MYOP      |
|3         |Bob         |ACNE DIAB100    |
|4         |George      |FLU             |
|5         |Marta       |DIAB150 ACNE FLU|
|6         |John        |HIGH_BP ASTHMA  |
|7         |Sarah       |DIAB2 FEVER     |
|8         |Robert      |COLD DIAB101    |
|9         |Emma        |HEART ACNE      |
|10        |David       |DIAB10 COUGH    |
+----------+------------+----------------+



# Using Spark SQL

In [4]:
patients.createOrReplaceTempView("patients")

In [11]:
spark.sql("""
    SELECT
        patient_id,
        patient_name,
        conditions
    FROM patients
    WHERE conditions RLIKE '(^| )DIAB1'
""").show()

# ^ -> Beginning of condition 
# | -> OR 
#  -> Condition starts with a space
# DIAB1 -> Prefix 

+----------+------------+----------------+
|patient_id|patient_name|      conditions|
+----------+------------+----------------+
|         2|       Alice|      DIAB1 MYOP|
|         3|         Bob|    ACNE DIAB100|
|         5|       Marta|DIAB150 ACNE FLU|
|         8|      Robert|    COLD DIAB101|
|        10|       David|    DIAB10 COUGH|
+----------+------------+----------------+



# Using Pyspark

In [18]:
from pyspark.sql.functions import col

result = patients.filter(
    col("conditions").rlike(r"(^| )DIAB1")
)

result.show()

+----------+------------+----------------+
|patient_id|patient_name|      conditions|
+----------+------------+----------------+
|         2|       Alice|      DIAB1 MYOP|
|         3|         Bob|    ACNE DIAB100|
|         5|       Marta|DIAB150 ACNE FLU|
|         8|      Robert|    COLD DIAB101|
|        10|       David|    DIAB10 COUGH|
+----------+------------+----------------+

